
# TP 03 — Collaborative Filtering

**Recommender Systems — SCIA — Session 03**

Theory reference: `03_CollaborativeFiltering_Theory.ipynb`.

Today you'll implement the collaborative filtering cost function (loop,
then vectorised), train it with a custom gradient descent loop, and
generate recommendations for yourself.

**Required standards:**
- modular code (small, reusable functions)
- numpy-style docstring on every function
- self-contained notebook (no dependency on other notebooks)


In [ ]:
import os

import numpy as np
import pandas as pd
import tensorflow as tf


## 1. Loading the dataset

This session uses a custom, pre-reduced subset of MovieLens (443 users,
4778 movies, focused on titles from 2000 onward) hosted on the course's
MEGA share, together with pre-computed `X`, `W`, `b` used to sanity-check
your cost function against known expected values before training on the
full thing.

Run the cell below once per machine — `megatools` needs to be installed
(`apt install megatools` / `brew install megatools`).


In [ ]:
%%bash
if [ ! -d "./data" ]; then
    megadl 'https://mega.nz/file/hEJhURAZ#8VeFh14WBrNnMvZxrOfrcXPpHwQDIssXJn8KTYFrPiQ'
    mkdir -p ./data/
    unzip -o movielens_small.zip -d ./data/
else
    echo "./data already present, skipping download."
fi



## 2. Loading the matrices

Write the loaders yourself — the expected CSV files are:
`small_movies_X.csv`, `small_movies_W.csv`, `small_movies_b.csv`,
`small_movies_Y.csv`, `small_movies_R.csv`, `small_movie_list.csv`.


In [ ]:
def load_precalc_params_small(
    data_dir: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, int, int, int]:
    """Load pre-computed X, W, b for sanity-checking the cost function.

    Parameters
    ----------
    data_dir : str
        Path to the directory containing ``small_movies_X.csv``,
        ``small_movies_W.csv`` and ``small_movies_b.csv``.

    Returns
    -------
    X : numpy.ndarray, shape (n_movies, n_features)
        Pre-computed movie feature vectors.
    W : numpy.ndarray, shape (n_users, n_features)
        Pre-computed user parameter vectors.
    b : numpy.ndarray, shape (1, n_users)
        Pre-computed user bias terms.
    n_movies : int
    n_features : int
    n_users : int

    Notes
    -----
    TODO: implement this function with np.loadtxt (delimiter=",").
    Remember b needs reshaping to (1, n_users).
    """
    raise NotImplementedError


def load_ratings_small(data_dir: str) -> tuple[np.ndarray, np.ndarray]:
    """Load the pre-built Y and R matrices for the small dataset.

    Parameters
    ----------
    data_dir : str
        Path to the directory containing ``small_movies_Y.csv`` and
        ``small_movies_R.csv``.

    Returns
    -------
    Y : numpy.ndarray, shape (n_movies, n_users)
    R : numpy.ndarray, shape (n_movies, n_users)

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


def load_movie_list(data_dir: str) -> tuple[list[str], pd.DataFrame]:
    """Load the movie titles, in the same row order as Y/R.

    Parameters
    ----------
    data_dir : str
        Path to the directory containing ``small_movie_list.csv``.

    Returns
    -------
    movie_titles : list of str
        Movie titles, ordered to match the rows of Y/R/X.
    movie_df : pandas.DataFrame
        The full table (title, mean rating, number of ratings, ...).

    Notes
    -----
    TODO: implement this function with pd.read_csv (header=0, index_col=0).
    """
    raise NotImplementedError

In [ ]:
data_dir = "./data"
X, W, b, num_movies, num_features, num_users = load_precalc_params_small(data_dir)
Y, R = load_ratings_small(data_dir)

print("Y", Y.shape, "R", R.shape)
print("X", X.shape, "W", W.shape, "b", b.shape)
print("num_features", num_features, "num_movies", num_movies, "num_users", num_users)


## 3. Exercise 1 — the cost function (loop implementation)

Implement `cofi_cost_func` using explicit `for` loops over users and
movies. Only accumulate the squared error where `R[i, j] == 1`.


In [ ]:
def cofi_cost_func(
    X: np.ndarray,
    W: np.ndarray,
    b: np.ndarray,
    Y: np.ndarray,
    R: np.ndarray,
    lambda_: float,
) -> float:
    """Compute the collaborative filtering cost, using explicit loops.

    Parameters
    ----------
    X : numpy.ndarray, shape (n_movies, n_features)
        Movie feature vectors.
    W : numpy.ndarray, shape (n_users, n_features)
        User parameter vectors.
    b : numpy.ndarray, shape (1, n_users)
        User bias terms.
    Y : numpy.ndarray, shape (n_movies, n_users)
        Observed ratings (0 where unobserved).
    R : numpy.ndarray, shape (n_movies, n_users)
        Indicator matrix: 1 where Y holds a real, observed rating.
    lambda_ : float
        Regularisation strength.

    Returns
    -------
    float
        The cost J(X, W, b) as defined in the theory notebook.

    Notes
    -----
    TODO: implement this function using two nested for loops (over users
    then movies), accumulating the squared error only where R[i, j] == 1,
    then adding the L2 regularisation term on W and X.
    """
    raise NotImplementedError


Test on a small slice of the real, pre-computed parameters — with known
expected values, so a mismatch tells you exactly where to look.


In [ ]:
n_users_r, n_movies_r, n_features_r = 4, 5, 3
X_r = X[:n_movies_r, :n_features_r]
W_r = W[:n_users_r, :n_features_r]
b_r = b[0, :n_users_r].reshape(1, -1)
Y_r = Y[:n_movies_r, :n_users_r]
R_r = R[:n_movies_r, :n_users_r]

cost_no_reg = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, lambda_=0)
print(f"Cost (no regularisation): {cost_no_reg:0.2f}   (expected: 13.67)")

cost_with_reg = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, lambda_=1.5)
print(f"Cost (with regularisation): {cost_with_reg:0.2f}   (expected: 28.09)")


## 4. Exercise 2 — the cost function (vectorised implementation)

Loops don't scale: the cost is recomputed at every training step, on the
full matrix. Implement a vectorised version using `tf.linalg.matmul`
instead.

**Watch out**: apply the `R` mask *before* squaring the error, or you'll
silently include every unobserved rating as if it were a real "0★" rating.


In [ ]:
def cofi_cost_func_vectorized(X, W, b, Y, R, lambda_: float):
    """Compute the collaborative filtering cost, vectorised.

    Parameters
    ----------
    X : array-like, shape (n_movies, n_features)
        Movie feature vectors (TensorFlow tensor or numpy array).
    W : array-like, shape (n_users, n_features)
        User parameter vectors.
    b : array-like, shape (1, n_users)
        User bias terms.
    Y : array-like, shape (n_movies, n_users)
        Observed ratings (0 where unobserved).
    R : array-like, shape (n_movies, n_users)
        Indicator matrix: 1 where Y holds a real, observed rating.
    lambda_ : float
        Regularisation strength.

    Returns
    -------
    float or tf.Tensor
        The cost J(X, W, b), numerically identical to `cofi_cost_func`
        given the same inputs.

    Notes
    -----
    TODO: implement this without Python-level loops, using matrix
    multiplication for the w·x term. Multiply the error by R before
    squaring it.
    """
    raise NotImplementedError

In [ ]:
cost_no_reg_v = cofi_cost_func_vectorized(X_r, W_r, b_r, Y_r, R_r, lambda_=0)
cost_with_reg_v = cofi_cost_func_vectorized(X_r, W_r, b_r, Y_r, R_r, lambda_=1.5)
print(f"Cost (no regularisation): {float(cost_no_reg_v):0.2f}   (expected: 13.67)")
print(f"Cost (with regularisation): {float(cost_with_reg_v):0.2f}   (expected: 28.09)")

assert np.isclose(
    float(cost_no_reg_v), cost_no_reg, atol=1e-6
), "Vectorised and loop implementations disagree (no regularisation) — check the R mask."
assert np.isclose(
    float(cost_with_reg_v), cost_with_reg, atol=1e-6
), "Vectorised and loop implementations disagree (with regularisation)."
print("Vectorised implementation matches the loop version.")


## 5. Rating some movies yourself

Look up a few movies you recognise in `small_movie_list.csv`, then rate
them. `my_ratings` is indexed by row position in `Y`/`movie_titles` — not
by an external movieId.


In [ ]:
def normalize_ratings(Y: np.ndarray, R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Subtract each movie's mean observed rating (mean-centre by row).

    Parameters
    ----------
    Y : numpy.ndarray, shape (n_movies, n_users)
        Ratings matrix.
    R : numpy.ndarray, shape (n_movies, n_users)
        Indicator matrix.

    Returns
    -------
    Y_norm : numpy.ndarray, shape (n_movies, n_users)
        Y with each row's mean (computed over observed entries only)
        subtracted.
    Y_mean : numpy.ndarray, shape (n_movies, 1)
        The per-movie mean that was subtracted (needed later to restore
        the original rating scale).

    Notes
    -----
    TODO: implement this function. Movies with zero ratings would divide
    by zero — guard against that (e.g. add a small epsilon to the count).
    """
    raise NotImplementedError

In [ ]:
movie_titles, movie_df = load_movie_list(data_dir)

# Find a few titles you recognise, note their row index, then rate them below.
[title for title in movie_titles if "Toy Story" in title]

In [ ]:
my_ratings = np.zeros(num_movies)

# Replace with row indices (from movie_titles) and ratings (0.5-5.0) that
# make sense to you. A couple of examples to get started:
# my_ratings[2700] = 5   # Toy Story 3 (2010)
# my_ratings[2609] = 2   # Persuasion (2007)

my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]
for i in my_rated:
    print(f"Rated {my_ratings[i]} for {movie_titles[i]}")

In [ ]:
# Fold your ratings in as an extra column, then normalise.
Y_ext = np.c_[my_ratings, Y]
R_ext = np.c_[(my_ratings != 0).astype(int), R]
Y_norm, Y_mean = normalize_ratings(Y_ext, R_ext)
print("Y_ext", Y_ext.shape)


## 6. Training

Initialise `X`, `W`, `b` randomly as `tf.Variable`s, then run the custom
`GradientTape` training loop from the theory notebook.


In [ ]:
num_movies_ext, num_users_ext = Y_ext.shape
n_features_train = 100
lambda_ = 1.0
iterations = 200

tf.random.set_seed(1234)
X_var = tf.Variable(
    tf.random.normal((num_movies_ext, n_features_train), dtype=tf.float64), name="X"
)
W_var = tf.Variable(
    tf.random.normal((num_users_ext, n_features_train), dtype=tf.float64), name="W"
)
b_var = tf.Variable(tf.random.normal((1, num_users_ext), dtype=tf.float64), name="b")

optimizer = tf.keras.optimizers.Adam(learning_rate=0.1)

for step in range(iterations):
    with tf.GradientTape() as tape:
        cost = cofi_cost_func_vectorized(X_var, W_var, b_var, Y_norm, R_ext, lambda_)
    grads = tape.gradient(cost, [X_var, W_var, b_var])
    optimizer.apply_gradients(zip(grads, [X_var, W_var, b_var]))
    if step % 20 == 0:
        print(f"iteration {step:>4}: cost = {float(cost):.1f}")


## 7. Recommendations

Predict every rating for your new-user column (column 0), restore the mean
that was subtracted in step 5, and rank the movies you haven't already
rated.


In [ ]:
def top_recommendations(
    X: np.ndarray,
    W: np.ndarray,
    b: np.ndarray,
    Y_mean: np.ndarray,
    movie_titles: list[str],
    movie_df: pd.DataFrame,
    my_rated: list[int],
    min_ratings: int = 20,
    n: int = 10,
) -> pd.DataFrame:
    """Rank the top-N recommended movies for the new user (column 0).

    Parameters
    ----------
    X, W, b : numpy.ndarray
        Learned parameters (call ``.numpy()`` on the tf.Variables first).
    Y_mean : numpy.ndarray, shape (n_movies, 1)
        Per-movie mean subtracted during normalisation, to be added back.
    movie_titles : list of str
        Movie title for each row, in order.
    movie_df : pandas.DataFrame
        Must contain a ``number of ratings`` column, aligned by row order
        with movie_titles, used for the min_ratings filter.
    my_rated : list of int
        Row indices already rated by this user; excluded from the results.
    min_ratings : int, optional
        Minimum number of ratings (across all users) a movie must have to
        be eligible (default 20).
    n : int, optional
        Number of recommendations to return (default 10).

    Returns
    -------
    pandas.DataFrame
        Columns: ``title``, ``predicted_rating``, ``number of ratings``,
        sorted descending by predicted rating, top ``n`` rows.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError

In [ ]:
recommendations = top_recommendations(
    X_var.numpy(),
    W_var.numpy(),
    b_var.numpy(),
    Y_mean,
    movie_titles=movie_titles,
    movie_df=movie_df,
    my_rated=my_rated,
    min_ratings=20,
    n=10,
)
recommendations


## 8. Comparison table

Log this session's model into the running comparison table from Session 01.


In [ ]:
def log_result(
    comparison_path: str, model_name: str, metric_name: str, metric_value: float
) -> pd.DataFrame:
    """Append a row to the running model-comparison table, creating it if needed.

    Parameters
    ----------
    comparison_path : str
        CSV path of the running comparison table.
    model_name : str
        Name of the model being logged.
    metric_name : str
        Name of the metric column.
    metric_value : float
        Value of the metric for this model.

    Returns
    -------
    pandas.DataFrame
        The updated comparison table.

    Notes
    -----
    TODO: implement this function (same pattern as Session 01's
    `log_baseline`).
    """
    raise NotImplementedError


log_result(
    comparison_path="./comparison_table.csv",
    model_name="Collaborative filtering (MF)",
    metric_name="mean_predicted_top10",
    metric_value=recommendations["predicted_rating"].mean(),
)


## 9. Going further (open-ended)

Pick at least one:

**Regularisation.** Retrain with three different values of `lambda_`
(e.g. 0, 1, 10). How do the recommendations change? Which value seems to
give the best balance between overfitting and underfitting, and how did
you judge that?

**Cold start.** Propose a strategy for recommending to a user with zero
ratings, or a movie nobody has rated yet. Prototype it — even a simple
heuristic counts — and discuss where it breaks down.

**Toward a hybrid.** Sketch (in words, or a few lines of code) how you'd
combine this model with Session 02's content-based filter. What would each
contribute? We'll build this properly in Session 05.

---
**End of TP 03.**
